#### Running Your Code in Colab

**Colab GPU Compute is limited!!!**

We strongly recommend you develop and test all of your code locally, and only upload this notebook to Colab to run the below cells once you feel confident in your implementation. **When you are not actively running a cell, you should disconnect your runtime** to save on compute.

Colab Pro is offered to students free of charge for 1 year. We strongly recommend you to utilize this offering. If you take more ML classes next semester, it will be extremely useful too.

If you are not using Colab Pro and you run out of compute, you will have to wait until the next day to try again. You should not leave this assignment until the last minute.

For experiments with a batch size greater than one, use a GPU in order to see a *significant* speedup. This is crucial in order to run the experiments in a reasonable amount of time.

You'll need to give Colab access to your code in one of two ways.

#### Option 1 (more secure): Importing code with a private GitHub Repository

Develop your code locally and push your work to a **private** GitHub repo.

Then, follow the steps to create a PAT and run the following cells

Steps to generate a GitHub Personal Access Token (PAT):
1. Follow this link: https://github.com/settings/personal-access-tokens/new
2. Give it a name
3. In Repository access, click "Only select repositories", and in the dropdown select your repository
4. In Permissions, click "Add permissions" and select "Contents". Keep the access as Read-only.
5. Generate the token and **copy it immediately**. You should save it somewhere on your computer.

You will paste that token into the input prompt below.

**Do not share your token with anyone.**

In [ ]:
# Prompt for repository URL and GitHub token
repo_url = input('Paste the full GitHub repository URL (e.g., https://github.com/username/repo): ').strip()
token = input('Paste your GitHub Personal Access Token: ').strip()

print('\nRepository URL recorded.')

In [ ]:
# Clone the repository using the token
import subprocess

# Insert token into URL
auth_repo_url = repo_url.replace('https://', f'https://{token}@')

print('Cloning repository...')
result = subprocess.run(['git', 'clone', auth_repo_url], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('Error during clone:')
    print(result.stderr)

In [ ]:
# Check whether the repository is private using the GitHub API
import requests

# Extract owner/repo from URL
parts = repo_url.rstrip('/').split('/')
owner, repo = parts[-2], parts[-1]

api_url = f'https://api.github.com/repos/{owner}/{repo}'

headers = {
    'Authorization': f'token {token}',
    'Accept': 'application/vnd.github+json'
}

response = requests.get(api_url, headers=headers)

if response.status_code == 200:
    repo_info = response.json()
    if repo_info.get('private'):
        print('Repository confirmed PRIVATE.')
    else:
        print('WARNING: This repository is PUBLIC. Please make it private as required.')
        print(f'Follow this link {repo_url+"/settings"}, scroll down to "Change repository visibility", and Change visibility.')
else:
    print('Could not verify repository visibility.')
    print('Status code:', response.status_code)


In [ ]:
# cd into the directory where your code is
# If you ran the previous cells correctly, you should be able to just run this cell
# If your repository contains folders, you may need to %cd into them
%cd "{repo}"

Note that colab will terminate after 60-90 minutes of inactivity, and will NOT save your work. Make sure to monitor your progress, and download any output metrics or model.pt files that you'll want to use for your emprical questions to avoid having to run the experiments again

#### Option 2: Uploading code and mounting Google Drive

Develop your code locally and upload your whole `/handout` folder to drive.
Then, run the following cells

In [ ]:
# Mount your drive to access files in your Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# cd into the directory where your code is
# e.g. %cd /content/drive/MyDrive/cmu/10-301/hw7/handout
%cd /content/drive/MyDrive/<path to your code>

#### Downloading data

In [ ]:
# You should only need to run this cell once. Comment it out once you have obtained these files.
# Make sure that you have a folder named 'data' in your directory
!mkdir data
!gdown --no-check-certificate "https://drive.google.com/uc?export=download&id=1YZhMGnJNLiB6mLdvlI0nW88CQg24X7Ku" -O data/HW7_large_stories.zip
!unzip data/HW7_large_stories.zip -d data

In [ ]:
# Set your data paths. Change this if the data is in a different location
tiny_train_stories = 'data/tiny_train_stories.json'
tiny_valid_stories = 'data/tiny_valid_stories.json'

full_train_stories = 'data/HW7_large_stories/train_stories.json'
full_valid_stories = 'data/HW7_large_stories/valid_stories.json'

#### Testing
Run the following cells on a CPU.

In [ ]:
!python test_runner.py

In [ ]:
!python rnn.py train --train_data {tiny_train_stories} --val_data {tiny_valid_stories} --embed_dim 64 --hidden_dim 128 --train_losses_out train_loss.txt --val_losses_out valid_loss.txt --metrics_out metrics.txt --dk 32 --dv 32 --num_sequences 128 --batch_size 1 --model_path model1.pt

#### Experiments
Run the following cells on a GPU.

#### 5.1

When training, uncomment the corresponding `embed_hidden_dims` and rerun the cell for each value.

In [ ]:
embed_hidden_dims = 64
# embed_hidden_dims = 128
# embed_hidden_dims = 256
# embed_hidden_dims = 512
!python rnn.py train --train_data {full_train_stories} --val_data {full_valid_stories} --embed_dim {embed_hidden_dims} --hidden_dim {embed_hidden_dims} --train_losses_out empiricals/e1/{embed_hidden_dims}_train_losses.txt --val_losses_out empiricals/e1/{embed_hidden_dims}_val_losses.txt --metrics_out empiricals/e1/{embed_hidden_dims}_metrics.txt --dk 128 --dv 128 --num_sequences 50000 --batch_size 128 --model_path embed_model.pt

Run this code to generate your plots for question 5.1

In [ ]:
!python plotter.py --question 1 --loss_folder empiricals/e1

#### 5.2

In [ ]:
batch_size = 32
# batch_size = 64
# batch_size = 128
# batch_size = 256
!python rnn.py train --train_data {full_train_stories} --val_data {full_valid_stories} --embed_dim 128 --hidden_dim 128 --train_losses_out empiricals/e2/{batch_size}_train_losses.txt --val_losses_out empiricals/e2/{batch_size}_val_losses.txt --metrics_out empiricals/e2/{batch_size}_metrics.txt --dk 128 --dv 128 --num_sequences 50000 --batch_size {batch_size} --model_path model_batch.pt

Run this code to generate your plots for question 5.2a

In [ ]:
!python plotter.py --question 2 --loss_folder empiricals/e2

#### 5.3

In [ ]:
num_sequences = 10000
# num_sequences = 20000
# num_sequences = 50000
# num_sequences = 100000
!python rnn.py train --train_data {full_train_stories} --val_data {full_valid_stories} --embed_dim 128 --hidden_dim 128 --train_losses_out empiricals/e3/{num_sequences}_train_losses.txt --val_losses_out empiricals/e3/{num_sequences}_val_losses.txt --metrics_out empiricals/e3/{num_sequences}_metrics.txt --dk 128 --dv 128 --num_sequences {num_sequences} --batch_size 128 --model_path model_sequences.pt

Run this code to generate your plots for question 5.3

In [ ]:
!python plotter.py --question 3 --loss_folder empiricals/e3

#### 5.4



In [ ]:
!python rnn.py train --train_data {full_train_stories} --val_data {full_valid_stories} --embed_dim 512 --hidden_dim 512 --train_losses_out empiricals/e4/train_losses.txt --val_losses_out empiricals/e4/val_losses.txt --metrics_out empiricals/e4/metrics.txt --dk 256 --dv 256 --num_sequences 250000 --batch_size 128 --model_path final_model.pt

In [ ]:
!python rnn.py generate --model_path final_model.pt --prefix "Once upon a time there was a" --temperature 0.0 --num_tokens 128

In [ ]:
!python rnn.py generate --model_path final_model.pt --prefix "Once upon a time there was a" --temperature 0.3 --num_tokens 128

In [ ]:
!python rnn.py generate --model_path final_model.pt --prefix "Once upon a time there was a" --temperature 0.8 --num_tokens 128